# UAS LiDAR demonstration

Search LiDAR items, load with laspy, extract a 0.5m wide transect and plot height information along the transect.

[Data used in this notebook](https://stac-api.tern.org.au/stac-browser/collections/uas__dronescape_lidar/items/20251123-SAAEYB0029.copc)


## Before you run
- Update placeholder values (`COLLECTION_ID`, dates, bounds, point coordinates) to match your data.
- Ensure auth is configured for protected assets (for example `.netrc` and/or GDAL config).
- Install optional dependencies required by this notebook's workflow (`laspy`, plotting extras).
- Run cells from top to bottom so variables are initialized in order.

### API Key

[Create API Key](https://ternaus.atlassian.net/wiki/spaces/TERNSup/pages/2353496065/Creating+API+Keys)

[Use API Key](https://ternaus.atlassian.net/wiki/spaces/TERNSup/pages/3355246599/Using+API+Keys+to+Access+TERN+Data+Services#Create-your-API-Key)

In [1]:
from tern_stac import TernStacClient, get_item_asset_href, laz_to_canopy_height, preview_raster
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import LineString, Point
import laspy
from laspy.copc import Bounds

# Define ID's for dataset we want to load

In [2]:
# Fill in from your catalog values
# Find the collection ID, item ID, and asset key from stac api browser
# https://stac-api.tern.org.au/stac-browser/?.language=en
COLLECTION_ID = "uas__dronescape_lidar"
POINT_CLOUD_ITEM_ID = "20251123-SAAEYB0029.copc"
POINT_CLOUD_ASSET_KEY = "20251123-SAAEYB0029.copc.laz"
# POINT_CLOUD_ITEM_ID = "20241208-SAAKAN0001.copc"
# POINT_CLOUD_ASSET_KEY = "20241208-SAAKAN0001.copc.laz"
POINT_CLOUD_MEDIA_TYPE = None
POINT_CLOUD_ROLE = None

# Use TERN-STAC to find data access urls

In [4]:
# Check collection for items
client = TernStacClient()
search = client.search(collections=[COLLECTION_ID])
items = list(search.items())
print(len(items), "Items found")

252 Items found


## Find the item we want in retrieved list of items

We want a very specfic item, identied by ID, and access the Cloud Optimised Point Cloud (COPC) file

In [5]:
item = None
for candidate in items:
    if candidate.id == POINT_CLOUD_ITEM_ID:
        item = candidate
        break

if item is None:
    raise RuntimeError("Item ID not found in results. Set POINT_CLOUD_ITEM_ID to a valid item id.")

asset_href = get_item_asset_href(
    item,
    asset_key=POINT_CLOUD_ASSET_KEY if POINT_CLOUD_ASSET_KEY != "" else None,
    media_type=POINT_CLOUD_MEDIA_TYPE,
    role=POINT_CLOUD_ROLE,
)
asset_href

'https://data.tern.org.au/uas/dronescape/SAAEYB0029/20251123/lidar/level1_proc/20251123-SAAEYB0029.copc.laz'

# Open Point Cloud with laspy COPC Reader

Define a transect 0.5m wide across the middle of the area

In [6]:
with laspy.CopcReader.open(asset_href) as crdr:

    xmin, ymin, zmin = crdr.header.mins
    xmax, ymax, zmax = crdr.header.maxs

    yc = (ymin + ymax) / 2

    # width | number of pints
    # 100 136120661
    # 50 64399874
    # 20 30193945
    # 10 18578445
    # 5 10185382
    # 2 4566053
    width = 0.5 # 1m ...
    bounds = Bounds(
        mins=(xmin, yc - width),
        maxs=(xmax, yc + width),
    )

    # Load only the points we are interested in
    pts = crdr.query(bounds=bounds)

print(len(pts), "Points extracted.")

1158361 Points extracted.


# Prepare plot data as vegetation curtain/profile plot

In [7]:
# Step 1: Project points onto transect (vectorised)
# X - horizontal position along the transect
# Y - how far each point is from transect centreline
# Z - vegetation height (elevation)
x = pts.x
y = pts.y
z = pts.z

# Distance along transect
dist_along = x - xmin

# Distance along transect
profile_dist = dist_along
# z - elevation
profile_height = z

## Plot the data

In [8]:
# Step 2: Nice point-cloud profile

# Plot is rendered as image to display in nbviewer.
import os
import matplotlib.pyplot as plt
# ensure target directory exists
os.makedirs("images", exist_ok=True)

with plt.ioff():
    fig, ax = plt.subplots(figsize=(14, 6))

    sc = ax.scatter(
        profile_dist,
        profile_height,
        c=profile_height, # use height as color
        cmap="turbo",
        s=1, # point size
        alpha=0.5, # opacity of each point (overlapping poinst accumulate visually ... dense regions become darker / more saturated)
    )

    plt.colorbar(sc, ax=ax, label="Elevation (m)")
    ax.set_xlabel("Distance along transect (m)")
    ax.set_ylabel("Elevation (m)")
    ax.set_title("LiDAR Vegetation Transect")

    plt.tight_layout()
    plt.savefig("images/04_UAS_Lidar.png")
    plt.close()

![UAS LiDAR elevation profile.](images/04_UAS_Lidar.png)
